Books to Scrape: Web Scraping, Cleaning and Analysis
Author: Oriomah Ogheneserome Gracious  |  Date:October 2026

Objective: Collect data on ~1,000 books from books.toscrape.com, clean it, analyze pricing and ratings patterns, and visualize the findings.

Tools: Python, requests, BeautifulSoup, pandas, matplotlib/seaborn

## 1. Import Libraries
Libraries for making web requests, parsing HTML, and handling data.

In [2]:
import time
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

## 2. Configuration and Session Setup
Reuse one connection for all requests and identify our scraper to the site

In [8]:
# Site settings
BASE = "https://books.toscrape.com/catalogue/"   # all book pages live under this address
LIST_URL = BASE + "page-{}.html"                 # {} gets replaced by the page number
DELAY = 0.3                                      # seconds to pause between requests
RATING_WORDS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

# Reuse one connection for all requests and identify our scraper to the site
session = requests.Session()
session.headers.update({"User-Agent": "capstone-project-data-collector"})

## 3. Scraping Functions
### 3.1 Request Helper
Fetches a page, handles errors and retries, and returns parsed HTML.

In [6]:
def get_soup(url, retries=3):
    """Fetch a page and return parsed HTML, or None if the page doesn't exist (404)."""
    for attempt in range(retries):
        try:
            r = session.get(url, timeout=30)
            if r.status_code == 404:          # page doesn't exist: we've passed the last page
                return None
            r.raise_for_status()              # turn other errors (500, 403...) into exceptions
            r.encoding = "utf-8"              # fixes "Â£" showing instead of "£"
            return BeautifulSoup(r.text, "html.parser")
        except requests.RequestException as e:
            print(f"  request failed ({e}), retry {attempt + 1}/{retries}")
            time.sleep(2 * (attempt + 1))     # wait longer after each failure
    raise RuntimeError(f"Giving up on {url}")

In [9]:
soup = get_soup(LIST_URL.format(1))
print(soup.title.text.strip())
print(len(soup.select("article.product_pod")), "books on page 1")

print(get_soup(LIST_URL.format(999)))   # should print None

All products | Books to Scrape - Sandbox
20 books on page 1
None


### 3.2 Listing Page Parser
Extracts the title, price, star rating, and link to the detail page for every book on a listing page.

In [10]:
def parse_listing_page(soup):
    """Return a list of dicts (one per book) from a listing page."""
    books = []
    for card in soup.select("article.product_pod"):                  # each book sits in one 'card'
        rating_class = card.select_one("p.star-rating")["class"]     # e.g. ['star-rating', 'Three']
        books.append({
            "title": card.h3.a["title"],                             # full title (visible text is cut short)
            "price": card.select_one("p.price_color").text.strip(),  # e.g. '£51.77'
            "rating": RATING_WORDS.get(rating_class[1]),             # 'Three' -> 3
            "url": urljoin(BASE, card.h3.a["href"]),                 # full link to the book's own page
        })
    return books

In [11]:
page1_books = parse_listing_page(soup)
print(len(page1_books), "books parsed")
print(page1_books[0])

pd.DataFrame(page1_books).head()

20 books parsed
{'title': 'A Light in the Attic', 'price': '£51.77', 'rating': 3, 'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}


,title,price,rating,url
0,A Light in the Attic,£51.77,3,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,£53.74,1,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,£50.10,1,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,£47.82,4,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,£54.23,5,https://books.toscrape.com/catalogue/sapiens-a...


### 3.3 Detail Page Parser
Extracts the category, UPC, tax details, stock availability, number of reviews, and description from an individual book's page.

In [12]:
def parse_detail_page(soup):
    """Return a dict of extra fields from a single book's page."""
    # Category is the 3rd link in the breadcrumb: Home > Books > Category > Title
    crumbs = soup.select("ul.breadcrumb li a")
    category = crumbs[2].text.strip() if len(crumbs) > 2 else None

    # The 'Product Information' table: left cell = label, right cell = value
    table = {row.th.text.strip(): row.td.text.strip()
             for row in soup.select("table.table-striped tr")}

    # The description paragraph sits right after the 'Product Description' heading
    desc_tag = soup.select_one("#product_description")
    description = desc_tag.find_next_sibling("p").text.strip() if desc_tag else None

    return {
        "category": category,
        "upc": table.get("UPC"),
        "price_excl_tax": table.get("Price (excl. tax)"),
        "price_incl_tax": table.get("Price (incl. tax)"),
        "tax": table.get("Tax"),
        "availability": table.get("Availability"),
        "num_reviews": table.get("Number of reviews"),
        "description": description,
    }

In [13]:
first_book_url = page1_books[0]["url"]
detail_soup = get_soup(first_book_url)
detail = parse_detail_page(detail_soup)
detail

{'category': 'Poetry',
 'upc': 'a897fe39b1053632',
 'price_excl_tax': '£51.77',
 'price_incl_tax': '£51.77',
 'tax': '£0.00',
 'availability': 'In stock (22 available)',
 'num_reviews': '0',
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius?

## 4. Run the Scraper
### 4.1 Collect all listing pages

In [14]:
all_books = []

for page in range(1, 51):                      # pages 1 to 50
    soup = get_soup(LIST_URL.format(page))
    if soup is None:                           # no more pages
        break
    all_books.extend(parse_listing_page(soup)) # add this page's 20 books
    time.sleep(DELAY)                          # short pause to be polite

df = pd.DataFrame(all_books)
Path("data/raw").mkdir(parents=True, exist_ok=True)
df.to_csv("data/raw/books_listing.csv", index=False)   # safety copy
print(df.shape)

(1000, 4)


### 4.2 Collect detail-page fields

In [15]:
details = []

for i, url in enumerate(df["url"], 1):
    soup = get_soup(url)
    details.append(parse_detail_page(soup) if soup else {})
    if i % 100 == 0:
        print(f"{i}/{len(df)} done")           # progress update every 100 books
    time.sleep(DELAY)

df_full = pd.concat([df, pd.DataFrame(details)], axis=1)   # glue the columns side by side
df_full.to_csv("data/raw/books_raw.csv", index=False)
print(df_full.shape)

  request failed (('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))), retry 1/3
100/1000 done
200/1000 done
300/1000 done
400/1000 done
500/1000 done
600/1000 done
700/1000 done
800/1000 done
900/1000 done
1000/1000 done
(1000, 12)


## 5. Data Cleaning
### 5.1 Inspect the raw data

In [16]:
df_full.info()                                  # column names, types, and missing values
print("Duplicate books:", df_full.duplicated(subset="upc").sum())
df_full.head(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   title           1000 non-null   object
 1   price           1000 non-null   object
 2   rating          1000 non-null   int64 
 3   url             1000 non-null   object
 4   category        1000 non-null   object
 5   upc             1000 non-null   object
 6   price_excl_tax  1000 non-null   object
 7   price_incl_tax  1000 non-null   object
 8   tax             1000 non-null   object
 9   availability    1000 non-null   object
 10  num_reviews     1000 non-null   object
 11  description     998 non-null    object
dtypes: int64(1), object(11)
memory usage: 93.9+ KB
Duplicate books: 0


,title,price,rating,url,category,upc,price_excl_tax,price_incl_tax,tax,availability,num_reviews,description
0,A Light in the Attic,£51.77,3,https://books.toscrape.com/catalogue/a-light-i...,Poetry,a897fe39b1053632,£51.77,£51.77,£0.00,In stock (22 available),0,It's hard to imagine a world without A Light i...
1,Tipping the Velvet,£53.74,1,https://books.toscrape.com/catalogue/tipping-t...,Historical Fiction,90fa61229261140a,£53.74,£53.74,£0.00,In stock (20 available),0,"""Erotic and absorbing...Written with starling ..."
2,Soumission,£50.10,1,https://books.toscrape.com/catalogue/soumissio...,Fiction,6957f44c3847a760,£50.10,£50.10,£0.00,In stock (20 available),0,"Dans une France assez proche de la nôtre, un h..."


### 5.2 Convert data types and tidy columns
- Prices: remove the "£" symbol and convert to numbers
- Stock: extract the number from "In stock (22 available)"
- Reviews: convert text to integers
- Drop redundant columns (tax fields)

In [17]:
df_clean = df_full.copy()                       # keep the raw data untouched

# Price: '£51.77' -> 51.77
df_clean["price"] = df_clean["price"].str.replace("£", "", regex=False).astype(float)

# Stock: 'In stock (22 available)' -> 22
df_clean["stock"] = df_clean["availability"].str.extract(r"(\d+)", expand=False).astype(int)

# Reviews: '0' -> 0
df_clean["num_reviews"] = df_clean["num_reviews"].astype(int)

# These columns repeat the price or are always £0.00
df_clean = df_clean.drop(columns=["price_excl_tax", "price_incl_tax", "tax", "availability"])

Path("data/clean").mkdir(parents=True, exist_ok=True)
df_clean.to_csv("data/clean/books_clean.csv", index=False)

df_clean.info()
df_clean.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        1000 non-null   object 
 1   price        1000 non-null   float64
 2   rating       1000 non-null   int64  
 3   url          1000 non-null   object 
 4   category     1000 non-null   object 
 5   upc          1000 non-null   object 
 6   num_reviews  1000 non-null   int64  
 7   description  998 non-null    object 
 8   stock        1000 non-null   int64  
dtypes: float64(1), int64(3), object(5)
memory usage: 70.4+ KB


,price,rating,num_reviews,stock
count,1000.00000,1000.000000,1000.0,1000.000000
mean,35.07035,2.923000,0.0,8.585000
std,14.44669,1.434967,0.0,5.654622
min,10.00000,1.000000,0.0,1.000000
25%,22.10750,2.000000,0.0,3.000000
50%,35.98000,3.000000,0.0,7.000000
75%,47.45750,4.000000,0.0,14.000000
max,59.99000,5.000000,0.0,22.000000


### 5.3 Remove uninformative columns and handle missing values
`num_reviews` is 0 for all 1,000 books, so it adds no information. Two books have no description; they are kept and labeled.

In [18]:
df_clean = df_clean.drop(columns=["num_reviews"])
df_clean["description"] = df_clean["description"].fillna("No description")
df_clean.to_csv("data/clean/books_clean.csv", index=False)
print(df_clean.shape, "| missing values:", df_clean.isna().sum().sum())

(1000, 8) | missing values: 0


Export the clean CSV

In [19]:
df_clean.to_csv("data/clean/books_clean.csv", index=False, encoding="utf-8-sig")
print(Path("data/clean/books_clean.csv").resolve())    # shows the full file location
print(df_clean.shape)                                  # should be (1000, 8)

C:\Users\HP\data\clean\books_clean.csv
(1000, 8)


In [20]:
print(Path("data/clean/books_clean.csv").resolve())


C:\Users\HP\data\clean\books_clean.csv
